# Matrix 

## Custom architecture of code for transformer of 200 million parameters

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Config

In [2]:
# Embedding dimension of transformer model
embedding_dim = 512
# feed forward network 
input_size = embedding_dim
hidden_size = embedding_dim * 5
output_size = embedding_dim

Gpt_config ={
    "vocab_size" : 50257, # vocubulary size
    "context_length" : 1024,
    "emb_dim" : 768 ,
    "num_heads" : 12,   # orignal 12 
    "num_layer" : 12,   # orignal 12 
    "drop_rate" : 0.1,
    "qkv_bias"  : False
}


### Feed forward network  

In [3]:
# feedforward function for the custom model
class FeedForward(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(FeedForward, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.selu = torch.nn.SiLU()
        self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
        self.selu = torch.nn.SiLU()
        self.fc3 = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x_1= self.fc1(x)
        out = self.selu(x_1)
        x_2 = self.fc2(out)
        out = self.selu(x_2)
        out = self.fc3(out)
        return out

In [4]:
# # visualize the feed forward network    
# from torchview import draw_graph
# model_ff = FeedForward(input_size, hidden_size, output_size)
# graph_ff = draw_graph(model_ff, input_size=(1, input_size))
# graph_ff.visual_graph

# Layer Normolization 

### RMSNorm (Root Mean Square Normalization)

RMSNorm is a simpler version of LayerNorm. It only rescales each token's vector by its root mean square. There is no mean subtraction and no bias, so it is faster and has fewer parameters.

$$\text{RMS}(x) = \sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}$$

$$\text{RMSNorm}(x)_i = \frac{x_i}{\text{RMS}(x)} \cdot g_i$$

where:
- $d$ = embedding dimension
- $g$ = learnable weight (one per feature)
- $\epsilon$ = small constant to avoid division by zero

Used in LLaMA, Mistral, Gemma and most modern LLMs.

In [5]:
# using RMSNorm (Root Mean Square Layer Normalization) 
class layer_norm(nn.Module):
    def __init__(self,embedding_dim,eps = 1e-5):
        super(layer_norm,self).__init__()
        self.eps = eps
        self.embedding_dim = embedding_dim
        self.weight = nn.Parameter(torch.ones(embedding_dim))
        
    def forward(self,x):
        rms_norm = nn.RMSNorm(self.embedding_dim, eps=self.eps)
        return rms_norm(x) * self.weight
        

# Multi-Head Attention Block

In [6]:
# MultiHeadAttention class for the custom model  written from scratch using nn.Module in PyTorch. It implements the multi-head attention mechanism used in transformer models. The class takes input dimensions, output dimensions, context length, number of heads, and an optional bias for query, key, and value projections. It computes the query, key, and value matrices, applies scaled dot-product attention with causal masking, and projects the output back to the desired dimension.
class MultiHeadAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length, num_heads,qkv_bias=False):
        super().__init__()
        assert d_in % num_heads == 0, "d_in must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_in // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        
    def forward(self, x):
        batch_size, seq_length, _ = x.size()
        # Compute Q, K, V
        Q = self.W_query(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.W_key(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.W_value(x).view(batch_size, seq_length, self.num_heads, self.head_dim).transpose(1, 2)

        """# Compute attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_scores = attn_scores.masked_fill(self.mask[:seq_length, :seq_length], float('-inf'))
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Compute output
        out = torch.matmul(attn_probs, V)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_out)
        out = self.out_proj(out)
        return out """
        
        out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_out)
        
        return self.out_proj(out)

## Transformer Block 

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.att = MultiHeadAttention(cfg["emb_dim"],
                                    cfg["emb_dim"],
                                    cfg["context_length"],
                                    cfg["num_heads"],
                                    cfg["qkv_bias"])
        self.feed_forward = FeedForward(cfg["emb_dim"], # input size
                                        cfg["emb_dim"] * 4, # hidden size
                                        cfg["emb_dim"]) # output size
        self.norm1 = layer_norm(cfg["emb_dim"])
        self.norm2 = layer_norm(cfg["emb_dim"])
        
    def forward(self,x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = x + shortcut
        
        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = x + shortcut
        return x

## custom model -- Matrix

In [8]:
# custom model (GPT-like) using the transformer block
## Tensor 

class Matrix(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        nn.init.normal_(self.tok_emb.weight, std=0.02)
        self.pos_emb = nn.Parameter(
            torch.randn(1, cfg["context_length"], cfg["emb_dim"]) * 0.02)
        
        self.trf_block = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layer"])]
            )
        
        self.norm = layer_norm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self,x):
        seq_length = x.size(1)
        
        tok_emb = self.tok_emb(x)
        pos_emb = self.pos_emb[:, :seq_length, :]
        
        x = tok_emb + pos_emb
        for block in self.trf_block:
            x = block(x)
        
        x = self.norm(x)
        logits = self.out_head(x)
        
        return logits




In [9]:
model = Matrix(Gpt_config)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# total model size 
total_size = total_params * 4 / (1024 ** 2)  # Assuming 4 bytes per parameter (float32)
print(f"Total model size: {total_size:.2f} MB")

Total number of parameters: 276,273,408
Total model size: 1053.90 MB


# Preparing datasets 

In [10]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="HuggingFaceFW/fineweb-edu",
#     repo_type="dataset",
#     allow_patterns="sample/10BT/00[0-5]_*.parquet",   # first 6 files
#     local_dir="/kaggle/working/fineweb_edu",
# )

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

enc = tiktoken.get_encoding("gpt2")
os.makedirs("/kaggle/working/tokens", exist_ok=True)

# 1. Tokenize each parquet shard once -> .bin (uint16)
for p in sorted(glob.glob("/kaggle/working/fineweb_edu/sample/10BT/*.parquet")):
    out = "/kaggle/working/tokens/" + os.path.basename(p).replace(".parquet", ".bin")
    if os.path.exists(out):
        continue
    texts = pd.read_parquet(p, columns=["text"])["text"].tolist()
    with open(out, "wb") as f:
        for i in range(0, len(texts), 10_000):
            docs = enc.encode_ordinary_batch(texts[i:i + 10_000], num_threads=4)
            ids = [t for d in docs for t in (*d, enc.eot_token)]
            np.asarray(ids, dtype=np.uint16).tofile(f)
    os.remove(p)   # free disk

bins = sorted(glob.glob("/kaggle/working/tokens/*.bin"))


# 2. Dataset: reads windows straight from the .bin files
class GPTDataset(Dataset):
    def __init__(self, bin_paths, max_length):
        self.max_length = max_length
        self.shards = [np.memmap(p, dtype=np.uint16, mode="r") for p in bin_paths]
        self.counts = [(len(s) - 1) // max_length for s in self.shards]  # windows per shard
        self.cum = np.cumsum([0] + self.counts)

    def __len__(self):
        return int(self.cum[-1])

    def __getitem__(self, idx):
        sid = int(np.searchsorted(self.cum, idx, side="right") - 1)
        start = int(idx - self.cum[sid]) * self.max_length
        chunk = self.shards[sid][start : start + self.max_length + 1]
        chunk = torch.from_numpy(chunk.astype(np.int64))
        return chunk[:-1], chunk[1:]          # input, target (shifted by 1)


# 3. Loaders
max_length = Gpt_config["context_length"]
batch_size = 8   # 1024 context + 12 layers: 32 will likely run out of memory on a T4

train_loader = DataLoader(GPTDataset(bins[:-1], max_length),
                          batch_size=batch_size,
                          shuffle=True,
                          drop_last=True,
                          num_workers=2)

val_loader   = DataLoader(GPTDataset(bins[-1:], max_length),
                          batch_size=batch_size,
                          shuffle=False,
                          drop_last=True,
                          num_workers=2)

x, y = next(iter(train_loader))
print(x.shape, y.shape)   # torch.Size([8, 1024]) each

In [ ]:
def calc_loss_batch(input_batch,target_batch,model,device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
    
    return loss 

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))  
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
            input_batch, target_batch, model, device)
            
            total_loss += loss.item()
        else:
            break
        
    return total_loss / num_batches


# Training Model

In [ ]:
import math
import torch

def get_lr(step, max_steps, warmup_steps, peak_lr, initial_lr=1e-6, min_lr=1e-5):
    """Learning rate for a given step: linear warmup, then cosine decay."""
    # Phase 1: warmup, go from initial_lr up to peak_lr in a straight line
    if step < warmup_steps:
        return initial_lr + (peak_lr - initial_lr) * step / warmup_steps

    # Phase 2: cosine decay, go from peak_lr down to min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)   # goes 0 -> 1
    cosine = 0.5 * (1 + math.cos(math.pi * progress))               # goes 1 -> 0
    return min_lr + (peak_lr - min_lr) * cosine

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, device,
                max_steps, warmup_steps, eval_freq, eval_iter,
                start_context, tokenizer,
                initial_lr=1e-6, min_lr=1e-5, max_grad_norm=1.0,
                ckpt_path="/kaggle/working/checkpoint.pt"):

    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen = 0
    step = 0

    # The lr you gave the optimizer is treated as the peak learning rate
    peak_lr = optimizer.param_groups[0]["lr"]

    model.train()
    while step < max_steps:                       # keep looping over the data until max_steps
        for input_batch, target_batch in train_loader:
            if step >= max_steps:
                break

            # 1. Set the learning rate for this step (warmup -> cosine decay)
            lr = get_lr(step, max_steps, warmup_steps, peak_lr, initial_lr, min_lr)
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
            track_lrs.append(lr)

            # 2. Forward pass + backward pass
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            # 3. Gradient clipping: cap the gradient norm to avoid exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

            # 4. Update the weights
            optimizer.step()
            tokens_seen += input_batch.numel()

            # 5. Every eval_freq steps: evaluate, print a sample, save a checkpoint
            if step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

                print(f"Step {step:06d} | lr {lr:.2e} | "
                      f"Train loss {train_loss:.3f} | Val loss {val_loss:.3f}")

                generate_and_print_sample(model, tokenizer, device, start_context)
                model.train()   # evaluation may have switched to eval mode

                torch.save({"model": model.state_dict(),
                            "optimizer": optimizer.state_dict(),
                            "step": step}, ckpt_path)

            step += 1

    return train_losses, val_losses, track_tokens_seen, track_lrs